In [ ]:
#!/usr/bin/env python3
"""
simple_voc_yolo_pipeline.py

Minimal single-file pipeline:
 - VOC XML -> YOLO txt labels
 - create data.yaml
 - clone yolov5 (if missing) and run training (fine-tune yolov5s)
 - parse training stdout to extract Precision / Recall / mAP metrics
 - run inference (pretrained or provided weights) and visualize/save predictions

Edit CONFIG section below to match your dataset paths.
Requires: git, python, torch, torchvision, opencv-python, matplotlib, pillow
"""

from pathlib import Path
import subprocess
import yaml
import re
import sys
import shutil
import os
from glob import glob
import xml.etree.ElementTree as ET
import cv2
import matplotlib.pyplot as plt
import torch

# ---------------------------
# CONFIG - edit these values
# ---------------------------
VOC_DIR = Path("./VOC")                     # folder containing Annotations/ and JPEGImages/
JPEG_DIR = VOC_DIR / "JPEGImages"           # image folder
LABELS_DIR = Path("./labels")               # output YOLO labels
CLASS_LIST = ["debris"]                     # class names (order -> class id)
YOLOV5_DIR = Path("./yolov5")               # where yolov5 repo will be cloned
DATA_YAML = Path("./data.yaml")             # produced data.yaml
IMG_SIZE = 640
BATCH = 16
EPOCHS = 20
WEIGHTS = "yolov5s.pt"                      # pretrained starting weights; can be changed to a local .pt
NOSAVE = True                               # default: do not save fine-tuned weights
SAVE_DIR = Path("./runs/train/exp")         # if NOSAVE=False, training will save here (yolov5 default)
TEST_IMAGE = None                           # path to one test image for inference e.g. "./test.jpg"
TEST_DIR = None                             # or directory of images for inference e.g. "./test_images"
OUT_VIS_DIR = Path("./preds")               # where visualized predictions will be saved (if set)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------

def voc_to_yolo(xml_path: Path, img_w: float, img_h: float):
    """
    Convert a single VOC XML to YOLO-format lines.
    Returns list of lines as strings.
    """
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in CLASS_LIST:
            continue
        cls_id = CLASS_LIST.index(name)
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        xmin = max(0.0, min(xmin, img_w))
        xmax = max(0.0, min(xmax, img_w))
        ymin = max(0.0, min(ymin, img_h))
        ymax = max(0.0, min(ymax, img_h))
        if xmax <= xmin or ymax <= ymin:
            continue
        x_center = ((xmin + xmax) / 2.0) / img_w
        y_center = ((ymin + ymax) / 2.0) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    return lines
    
def convert_voc_folder(voc_dir: Path, jpeg_dir: Path, labels_dir: Path):
    """
    Convert all XMLs under voc_dir (Annotations) to YOLO .txt under labels_dir
    with the same relative structure as jpeg_dir.
    """
    xml_paths = sorted(voc_dir.rglob("*.xml"))
    print(f"[convert] found {len(xml_paths)} XML files")
    for xml in xml_paths:
        base = xml.stem
        # find image under jpeg_dir
        candidate = None
        for ext in (".jpg", ".jpeg", ".png"):
            p = jpeg_dir / (base + ext)
            if p.exists():
                candidate = p
                break
        if candidate is None:
            imgs = list(jpeg_dir.rglob(base + ".*"))
            candidate = imgs[0] if imgs else None
        if candidate is None:
            print(f"[convert] skip {xml.name}: matching image not found")
            continue
        # get image size
        img = cv2.imread(str(candidate))
        h, w = img.shape[:2]
        rel = candidate.relative_to(jpeg_dir)
        out_txt = labels_dir / rel.with_suffix(".txt")
        out_txt.parent.mkdir(parents=True, exist_ok=True)
        lines = voc_to_yolo(xml, w, h)
        out_txt.write_text("\n".join(lines))
    print("[convert] finished")

def write_data_yaml(train_img_dir: Path, val_img_dir: Path, test_img_dir: Path, out_path: Path):
    content = {
        "train": str(train_img_dir.resolve()),
        "val": str(val_img_dir.resolve()) if val_img_dir else str(train_img_dir.resolve()),
        "test": str(test_img_dir.resolve()) if test_img_dir else str(train_img_dir.resolve()),
        "nc": len(CLASS_LIST),
        "names": CLASS_LIST
    }
    out_path.write_text(yaml.safe_dump(content))
    print(f"[data.yaml] written to {out_path}")

def git_clone_yolov5(dest: Path):
    if dest.exists():
        print(f"[git] yolov5 already cloned at {dest}")
        return
    subprocess.run(["git", "clone", "https://github.com/ultralytics/yolov5.git", str(dest)], check=True)
    print("[git] cloned yolov5")

def run_yolov5_train(yolov5_dir: Path, data_yaml: Path, weights: str, imgsz:int, batch:int, epochs:int, nosave: bool):
    cmd = [sys.executable, str(yolov5_dir / "train.py"),
           "--img", str(imgsz), "--batch", str(batch), "--epochs", str(epochs),
           "--data", str(data_yaml), "--weights", str(weights)]
    if nosave:
        cmd += ["--nosave"]
    print("[train] running:", " ".join(cmd))
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = proc.stdout
    print(out)
    return out

def parse_metrics_from_yolov5_output(stdout_text: str):
    """
    Extract Precision, Recall, mAP@0.5 and mAP@0.5:0.95 from train.py stdout.
    Returns dict with keys: precision, recall, map50, map5095
    """
    metrics = {"precision": None, "recall": None, "map50": None, "map5095": None}
    # look for lines containing val/... or "Precision" or "mAP"
    # common YOLOv5 summary pattern contains: "val/precision: 0.69 val/recall: 0.78 val/mAP_0.5: 0.70 val/mAP_0.5:0.95: 0.38"
    pattern = re.compile(r"val/precision:\s*([0-9.]+).*?val/recall:\s*([0-9.]+).*?val/mAP_0.5:\s*([0-9.]+).*?val/mAP_0.5:0.95:\s*([0-9.]+)", re.S)
    m = pattern.search(stdout_text)
    if m:
        metrics["precision"] = float(m.group(1))
        metrics["recall"] = float(m.group(2))
        metrics["map50"] = float(m.group(3))
        metrics["map5095"] = float(m.group(4))
        return metrics
    # fallback patterns
    p2 = re.compile(r"Precision\s*:\s*([0-9.]+)\s*Recall\s*:\s*([0-9.]+)\s*mAP@.50\s*:\s*([0-9.]+)", re.I)
    m2 = p2.search(stdout_text)
    if m2:
        metrics["precision"] = float(m2.group(1))
        metrics["recall"] = float(m2.group(2))
        metrics["map50"] = float(m2.group(3))
    return metrics

def load_model_for_inference(weights_path: str = None, device: str = DEVICE):
    if weights_path and Path(weights_path).exists():
        model = torch.hub.load("ultralytics/yolov5", "custom", str(weights_path))
    else:
        model = torch.hub.load("ultralytics/yolov5", "yolov5s", pretrained=True)
    model.to(device)
    model.eval()
    return model

def visualize_predictions(img_path: Path, detections, names, save_path: Path = None, show: bool = True):
    img = cv2.imread(str(img_path))
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        x1,y1,x2,y2 = map(int, (x1,y1,x2,y2))
        label = f"{names[int(cls)]} {conf:.2f}" if names else f"{int(cls)} {conf:.2f}"
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,165,255), 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, y1-th-6), (x1+tw, y1), (0,165,255), -1)
        cv2.putText(img, label, (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1)
    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), img)
    if show:
        plt.figure(figsize=(10,8)); plt.axis('off'); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.show()

def run_inference_and_visualize(model, images, out_dir: Path = None, imgsz:int=IMG_SIZE):
    names = model.names if hasattr(model, "names") else None
    for p in images:
        res = model(str(p), size=imgsz)
        detections = []
        for row in res.xyxy[0].cpu().numpy():
            x1,y1,x2,y2,conf,cls = row.tolist()
            detections.append([x1,y1,x2,y2,conf,int(cls)])
        save_path = out_dir / Path(p).name if out_dir else None
        visualize_predictions(Path(p), detections, names, save_path, show=True)
        print(f"[infer] {Path(p).name}: {len(detections)} detections")

# ------------------------
# Main pipeline
# ------------------------
def main():
    # 1) Convert VOC -> YOLO labels
    convert_voc_folder(VOC_DIR, JPEG_DIR, LABELS_DIR)

    # 2) Prepare data.yaml (train/val/test point to image folders)
    # By default point to JPEG_DIR for train/val/test; user may edit DATA_YAML manually if needed.
    write_data_yaml(JPEG_DIR, None, None, DATA_YAML)

    # 3) Clone yolov5 if missing
    if not YOLOV5_DIR.exists():
        print("[setup] cloning yolov5 repo (this requires internet and git)")
        subprocess.run(["git","clone","https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(YOLOV5_DIR / "requirements.txt")], check=True)
    else:
        print("[setup] yolov5 already present")

    # 4) Run training (fine-tune yolov5s)
    out = run_yolov5_train(YOLOV5_DIR, DATA_YAML, WEIGHTS, IMG_SIZE, BATCH, EPOCHS, NOSAVE)

    # 5) Parse metrics from training stdout
    metrics = parse_metrics_from_yolov5_output(out)
    print("[metrics] extracted:", metrics)

    # 6) Inference: if fine-tuned weights saved and you want to use them, set weights_path accordingly.
    # Because NOSAVE=True by default, the script will use pretrained yolov5s for inference.
    weights_for_infer = None
    if not NOSAVE:
        # expected saved path (yolov5 default) -> use SAVE_DIR/weights/best.pt or last.pt
        maybe_best = SAVE_DIR / "weights" / "best.pt"
        maybe_last  = SAVE_DIR / "weights" / "last.pt"
        if maybe_best.exists():
            weights_for_infer = str(maybe_best)
        elif maybe_last.exists():
            weights_for_infer = str(maybe_last)
    print("[inference] loading model for inference. using:", weights_for_infer or "pretrained yolov5s")

    model = load_model_for_inference(weights_for_infer)

    # 7) Gather test images
    images = []
    if TEST_IMAGE:
        images = [TEST_IMAGE]
    elif TEST_DIR:
        images = sorted([str(p) for p in Path(TEST_DIR).rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png")])
    else:
        # fallback - show a couple of images from JPEG_DIR
        images = sorted([str(p) for p in JPEG_DIR.rglob("*.jpg")])[:4]

    images = images[:50]   # limit to first 50 images to be practical
    print(f"[inference] running on {len(images)} image(s)")

    run_inference_and_visualize(model, images, out_dir=OUT_VIS_DIR if OUT_VIS_DIR else None, imgsz=IMG_SIZE)

    print("[done] pipeline complete. Note: fine-tuned model will not be available for inference unless NOSAVE=False and weights were saved.")

if __name__ == "__main__":
    main()